In [1]:
import SimpleITK as sitk
from readii.negative_controls import (
  getArrayFromImageOrArray,
  makeShuffleImage,
  makeRandomImage,
  makeRandomSampleFromDistributionImage,
  negativeControlROIOnly,
  negativeControlNonROIOnly,
  applyNegativeControl
)

from pathlib import Path
from readii.image_processing import (
  loadDicomSITK,
  loadRTSTRUCTSITK,
  loadSegmentation
)
from readii.image_processing import alignImages, getROIVoxelLabel

In [ ]:
# # Check the CT image
# print(f"CT Image Size: {ct.GetSize()}")
# print(f"CT Image Spacing: {ct.GetSpacing()}")
# print(f"CT Image Origin: {ct.GetOrigin()}")
# print(f"CT Image Direction: {ct.GetDirection()}")

# # Check the segmentation image
# print(f"Segmentation Image Size: {seg.GetSize()}")
# print(f"Segmentation Image Spacing: {seg.GetSpacing()}")
# print(f"Segmentation Image Origin: {seg.GetOrigin()}")
# print(f"Segmentation Image Direction: {seg.GetDirection()}")

NameError: name 'ct' is not defined

In [22]:
import SimpleITK as sitk
import numpy as np
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
from functools import partial
# Define the size and center of the image
size = [128, 128, 128]
center = [int(s / 2) for s in size]
radius = 50
square_side = 10  # Side length of the square
offset = [10, -10, 0]  # Offset from the center [x_offset, y_offset, z_offset]

# Create an empty image for the sphere
original_sphere = sitk.Image(size, sitk.sitkFloat32)

# Create an empty image for the segmentation
segmentation_image = sitk.Image(size, sitk.sitkUInt8)

# Iterate through each voxel and assign values based on the distance from the center
for z in range(size[2]):
    for y in range(size[1]):
        for x in range(size[0]):
            distance = np.sqrt((x - center[0]) ** 2 + (y - center[1]) ** 2 + (z - center[2]) ** 2)
            if distance <= radius:
                original_sphere.SetPixel(x, y, z, 1 - (distance / radius))
                
                # Define square boundaries
                x_min = center[0] + offset[0] - square_side // 2
                x_max = center[0] + offset[0] + square_side // 2
                y_min = center[1] + offset[1] - square_side // 2
                y_max = center[1] + offset[1] + square_side // 2
                z_min = center[2] + offset[2] - square_side // 2
                z_max = center[2] + offset[2] + square_side // 2

                # Add segmentation if within the square boundaries
                if x_min <= x <= x_max and y_min <= y <= y_max and z_min <= z <= z_max:
                    segmentation_image.SetPixel(x, y, z, 1)
            else:
                original_sphere.SetPixel(x, y, z, 0)



In [23]:
# Function to plot a slice of the image
def plot_slice(sphere_image, segmentation_image, z):
    plt.figure(figsize=(6, 6))
    sphere_slice = sitk.GetArrayViewFromImage(sphere_image)[z, :, :]
    segmentation_slice = sitk.GetArrayViewFromImage(segmentation_image)[z, :, :]

    # Plot the sphere with segmentation overlay
    plt.imshow(sphere_slice, cmap='gray')
    plt.imshow(segmentation_slice, cmap='Reds', alpha=0.5)  # Overlay the segmentation
    plt.title(f'Slice at z={z}')
    plt.axis('off')
    plt.show()


In [24]:

# Create a slider widget
z_slider = widgets.IntSlider(min=0, max=size[2]-1, step=1, value=center[2])
widgets.interact(lambda z: plot_slice(original_sphere, segmentation_image, z), z=z_slider)

display(z_slider)

interactive(children=(IntSlider(value=64, description='z', max=127), Output()), _dom_classes=('widget-interact…

IntSlider(value=64, description='z', max=127)

In [25]:

# Create a slider widget
z_slider = widgets.IntSlider(min=0, max=size[2]-1, step=1, value=center[2])
sphere_image = applyNegativeControl(original_sphere, roiMask=segmentation_image, negativeControlType="shuffled", negativeControlRegion="full")

widgets.interact(lambda z: plot_slice(sphere_image, segmentation_image, z), z=z_slider)

display(z_slider)

interactive(children=(IntSlider(value=64, description='z', max=127), Output()), _dom_classes=('widget-interact…

IntSlider(value=64, description='z', max=127)

In [26]:

# Create a slider widget
z_slider = widgets.IntSlider(min=0, max=size[2]-1, step=1, value=center[2])
sphere_image = applyNegativeControl(original_sphere, roiMask=segmentation_image, negativeControlType="shuffled", negativeControlRegion="non_roi")

widgets.interact(lambda z: plot_slice(sphere_image, segmentation_image, z), z=z_slider)

display(z_slider)

interactive(children=(IntSlider(value=64, description='z', max=127), Output()), _dom_classes=('widget-interact…

IntSlider(value=64, description='z', max=127)

In [27]:

# Create a slider widget
z_slider = widgets.IntSlider(min=0, max=size[2]-1, step=1, value=center[2])
sphere_image = applyNegativeControl(original_sphere, roiMask=segmentation_image, negativeControlType="shuffled", negativeControlRegion="roi")

widgets.interact(lambda z: plot_slice(sphere_image, segmentation_image, z), z=z_slider)

display(z_slider)

interactive(children=(IntSlider(value=64, description='z', max=127), Output()), _dom_classes=('widget-interact…

IntSlider(value=64, description='z', max=127)

In [28]:

# Create a slider widget
z_slider = widgets.IntSlider(min=0, max=size[2]-1, step=1, value=center[2])
sphere_image = applyNegativeControl(original_sphere, roiMask=segmentation_image, negativeControlType="randomized", negativeControlRegion="full")

widgets.interact(lambda z: plot_slice(sphere_image, segmentation_image, z), z=z_slider)

display(z_slider)

interactive(children=(IntSlider(value=64, description='z', max=127), Output()), _dom_classes=('widget-interact…

IntSlider(value=64, description='z', max=127)

In [29]:

# Create a slider widget
z_slider = widgets.IntSlider(min=0, max=size[2]-1, step=1, value=center[2])
sphere_image = applyNegativeControl(original_sphere, roiMask=segmentation_image, negativeControlType="randomized", negativeControlRegion="non_roi")

widgets.interact(lambda z: plot_slice(sphere_image, segmentation_image, z), z=z_slider)

display(z_slider)

interactive(children=(IntSlider(value=64, description='z', max=127), Output()), _dom_classes=('widget-interact…

IntSlider(value=64, description='z', max=127)

In [ ]:

# Create a slider widget
z_slider = widgets.IntSlider(min=0, max=size[2]-1, step=1, value=center[2])
sphere_image = applyNegativeControl(original_sphere, roiMask=segmentation_image, negativeControlType="randomized", negativeControlRegion="roi")

widgets.interact(lambda z: plot_slice(sphere_image, segmentation_image, z), z=z_slider)

display(z_slider)

interactive(children=(IntSlider(value=64, description='z', max=127), Output()), _dom_classes=('widget-interact…

IntSlider(value=64, description='z', max=127)